# Knowledge Distillation

In knowledge distillation (KD), a **teacher** network transfers information to a smaller **student** network.


## 1. What information can a teacher transfer?

A labeled image may have the target **cat**, but the teacher predicts a full distribution:

| Class | Example probability |
|:--|--:|
| Cat | 0.75 |
| Dog | 0.20 |
| Deer | 0.04 |
| Truck | 0.01 |

The probabilities describe which alternatives the teacher considers plausible. This extra information is called *dark knowledge*.

| Method | Teacher information | Student objective |
|:--|:--|:--|
| Supervised baseline | None | Predict the correct class |
| Hinton KD | Soft probabilities | Match the teacher's class distribution |
| Feature distillation | Intermediate feature maps | Match teacher representations |
| Logit matching | Raw class scores (logits) | Regress student logits toward teacher logits |
| Combined KD (optional) | Probabilities and features | Minimize both kinds of losses |
| Regression KD (Part II) | Continuous numerical predictions | Regress student outputs toward teacher outputs |

The teacher is trained first and then **frozen**. During distillation, teacher predictions are targets, and only the student (plus any student-side adapter) is optimized.

```text
Input ──► Frozen teacher ──► predictions and/or features ──┐
                                                          ├──► Loss ──► gradients to student
Input ──► Trainable student ─► predictions and/or features ┘
```

We use the same student architecture, data split, and training schedule for all CIFAR-10 methods so that the main difference is the information transferred from the teacher.

**Why not use hard pseudo-labels as a main method here?** Taking `argmax` discards the teacher's probability distribution. On fully labeled CIFAR-10 it usually repeats the known label when the teacher is correct and adds label noise when the teacher is wrong. Hard pseudo-labels can still be useful on unlabeled data, but that is a different learning setup.


## 2. Imports and experiment settings


- `TEMPERATURE`: how much to soften the class probabilities in Hinton KD.
- `ALPHA`: relative weight for ground-truth cross-entropy in Hinton KD.
- `FEATURE_WEIGHT`: weight of intermediate feature matching.
- `LOGIT_WEIGHT`: weight of the logit regression loss.

Loss weights are hyperparameters. A value that works for one teacher or architecture may not work for another.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Device:', device)

CHECKPOINT_DIR = Path('./kd_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Short experiments; increase the epoch counts for stronger models.
BATCH_SIZE = 128
TEACHER_EPOCHS = 10
STUDENT_EPOCHS = 8
TEMPERATURE = 4.0
ALPHA = 0.5               # weight for hard-label CE in Hinton KD
FEATURE_WEIGHT = 1.0      # weight for normalized feature regression
LOGIT_WEIGHT = 0.05      # weight for matching raw logits with MSE

TRAIN_TEACHER = True       # set False to reuse the saved teacher checkpoint
# Optional experiment at the end of the training sections.
RUN_COMBINED = False


## 3. Dataset: CIFAR-10

CIFAR-10 contains 60,000 colour images, each **32 × 32 pixels**, from 10 classes. We use the official 50,000-image training portion for training and validation, and reserve the official 10,000-image test portion for the final comparison.

Our split is:

- **45,000 training images:** used to update model parameters.
- **5,000 validation images:** used to choose the best checkpoint after each epoch.
- **10,000 test images:** used only to report final accuracy.

During training, random crops and flips make the task slightly harder and improve generalization. Validation/test images must be transformed **without randomness**; otherwise the validation result can change simply because an image was cropped differently.

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2430, 0.2610)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

DATA_DIR = './data'
all_train_aug = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_transform)
all_train_eval = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=eval_transform)

indices = torch.randperm(len(all_train_aug), generator=torch.Generator().manual_seed(SEED))
val_indices = indices[:5_000].tolist()
train_indices = indices[5_000:].tolist()
train_dataset = Subset(all_train_aug, train_indices)
val_dataset = Subset(all_train_eval, val_indices)

NUM_WORKERS = 2 if torch.cuda.is_available() else 0
loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=(device.type == 'cuda'))
train_rng = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=train_rng, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)

classes = all_train_aug.classes
print('Classes:', classes)
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}')


### Inspect a few training images

The images below come from the training split **without random augmentation**. Their class names are taken from the dataset labels.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, idx in zip(axes.flat, train_indices[:10]):
    image, target = all_train_eval[idx]
    rgb = image * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)
    ax.imshow(rgb.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(classes[target], fontsize=10)
    ax.axis('off')
plt.suptitle('CIFAR-10: training examples', y=1.02)
plt.tight_layout()
plt.show()

## 4. Models: a larger teacher and a smaller student

The **teacher** is a ResNet-18. We adapt its first convolution for 32 × 32 images, then train it on CIFAR-10.

The **student** is a smaller convolutional network with three blocks. Its final classifier takes a 128-dimensional vector obtained by global average pooling.

For feature distillation, we also need the networks' intermediate outputs:

| Network | Selected feature map | Explanation |
|:--|:--|:--|
| ResNet-18 teacher | `[B, 512, 4, 4]` | Output of its final residual stage |
| Small CNN student | `[B, 128, 8, 8]` | Output of its last convolutional block |

`B` is the batch size, `C` is the number of channels, and `H, W` are the spatial dimensions. These shapes are different, so later we will learn a projection that makes the student features comparable to the teacher features.

Run the next cell and inspect the **parameter counts** and **feature shapes** printed below the models.

In [ ]:
def make_teacher(num_classes=10):
    model = models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


class StudentCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, num_classes))

    def forward_with_features(self, x):
        feature_map = self.encoder(x)
        logits = self.classifier(self.pool(feature_map))
        return logits, feature_map

    def forward(self, x):
        logits, _ = self.forward_with_features(x)
        return logits


def teacher_forward_with_features(model, x):
    '''A ResNet-18 forward pass that also exposes the last convolutional feature map.'''
    x = model.relu(model.bn1(model.conv1(x)))
    x = model.maxpool(x)
    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    feature_map = model.layer4(x)
    logits = model.fc(torch.flatten(model.avgpool(feature_map), 1))
    return logits, feature_map


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

teacher = make_teacher(len(classes)).to(device)
student_example = StudentCNN(len(classes)).to(device)
print(f'Teacher parameters: {count_parameters(teacher):,}')
print(f'Student parameters: {count_parameters(student_example):,}')

# Check intermediate shapes and that our manual teacher forward matches its normal forward.
teacher.eval()
student_example.eval()
with torch.no_grad():
    x_demo = torch.randn(2, 3, 32, 32, device=device)
    teacher_logits, teacher_features = teacher_forward_with_features(teacher, x_demo)
    student_logits, student_features = student_example.forward_with_features(x_demo)
    torch.testing.assert_close(teacher_logits, teacher(x_demo))
    print('Teacher:', tuple(teacher_features.shape), '| Student:', tuple(student_features.shape))
del x_demo, teacher_logits, teacher_features, student_logits, student_features


## 5. Train the teacher

Distillation requires a teacher that has learned something useful. Here, we train ResNet-18 with **cross-entropy loss**:

\[
\mathcal L_{\mathrm{CE}}=-\log p_{\theta}(y\mid x).
\]

For each training image \(x\), the model should assign high probability to its true class \(y\).

At the end of every epoch, we measure **validation accuracy**. We save the checkpoint with the best validation accuracy, then load it and freeze the teacher.

**Practical tip:** after successfully training the teacher, set `TRAIN_TEACHER = False` in Section 2 to reuse its saved weights when re-running the notebook. The checkpoint is stored in the notebook's working directory. In Colab, this directory may be cleared when the runtime resets.

In [ ]:
def run_supervised_epoch(model, loader, optimizer=None, description='Train'):
    training = optimizer is not None
    model.train(training)
    loss_sum = 0.0
    correct = 0
    n = 0
    iterator = tqdm(loader, desc=description, leave=False) if training else loader
    with torch.set_grad_enabled(training):
        for images, targets in iterator:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
            batch = targets.size(0)
            loss_sum += loss.item() * batch
            correct += (logits.argmax(dim=1) == targets).sum().item()
            n += batch
            if training:
                iterator.set_postfix(loss=f'{loss_sum/n:.3f}', acc=f'{100*correct/n:.1f}%')
    return {'loss': loss_sum / n, 'acc': correct / n}


TEACHER_PATH = CHECKPOINT_DIR / 'teacher_resnet18_cifar10.pth'
if TRAIN_TEACHER:
    optimizer = torch.optim.SGD(teacher.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TEACHER_EPOCHS)
    best_teacher_acc = -1.0
    teacher_history = []
    for epoch in range(1, TEACHER_EPOCHS + 1):
        train_metrics = run_supervised_epoch(teacher, train_loader, optimizer, f'Teacher {epoch}/{TEACHER_EPOCHS}')
        val_metrics = run_supervised_epoch(teacher, val_loader)
        teacher_history.append((train_metrics, val_metrics))
        if val_metrics['acc'] > best_teacher_acc:
            best_teacher_acc = val_metrics['acc']
            torch.save(teacher.state_dict(), TEACHER_PATH)
        scheduler.step()
        print(f'Teacher {epoch:2d}: train acc {train_metrics["acc"]:.3f} | '
              f'val acc {val_metrics["acc"]:.3f} | best {best_teacher_acc:.3f}')

if not TEACHER_PATH.is_file():
    raise FileNotFoundError(f'Missing teacher checkpoint: {TEACHER_PATH}. Set TRAIN_TEACHER=True.')
teacher.load_state_dict(torch.load(TEACHER_PATH, map_location=device, weights_only=True))
teacher.eval()
teacher.requires_grad_(False)
teacher_val = run_supervised_epoch(teacher, val_loader)
print(f'Frozen teacher validation accuracy: {teacher_val["acc"]:.2%}')


## 6. Baseline: supervised student


In [ ]:
# All students start with identical weights.
initial_student = StudentCNN(len(classes))
initial_student_state = {
    name: value.detach().clone() for name, value in initial_student.state_dict().items()
}
del initial_student


def fresh_student():
    """Make a new student with the same initial weights and data order."""
    random.seed(SEED + 100)
    np.random.seed(SEED + 100)
    torch.manual_seed(SEED + 100)
    train_rng.manual_seed(SEED + 100)
    student = StudentCNN(len(classes)).to(device)
    student.load_state_dict(initial_student_state)
    return student


def update_student_history(name, epoch, student, history, best_acc, checkpoint,
                           train_loss, train_acc, adapter=None, **loss_terms):
    """Track validation metrics and save the best checkpoint."""
    validation = run_supervised_epoch(student, val_loader)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': validation['loss'],
        'val_acc': validation['acc'],
        **loss_terms,
    })
    if validation['acc'] > best_acc:
        best_acc = validation['acc']
        torch.save({
            'student': student.state_dict(),
            'adapter': adapter.state_dict() if adapter is not None else None,
        }, checkpoint)
    print(f'{name}: epoch {epoch}/{STUDENT_EPOCHS}, '
          f'train acc {train_acc:.2%}, val acc {validation["acc"]:.2%}, '
          f'best val acc {best_acc:.2%}')
    return best_acc


def restore_best_student(name, student, checkpoint, history, best_acc, adapter=None):
    saved = torch.load(checkpoint, map_location=device, weights_only=True)
    student.load_state_dict(saved['student'])
    student.eval()
    if adapter is not None:
        adapter.load_state_dict(saved['adapter'])
        adapter.eval()
    return {
        'student': student,
        'adapter': adapter,
        'history': pd.DataFrame(history),
        'best_val_acc': best_acc,
        'checkpoint': str(checkpoint),
        'method': name,
    }


experiments = {}

In [ ]:
def train_supervised_student():
    student = fresh_student()
    optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STUDENT_EPOCHS)
    checkpoint = CHECKPOINT_DIR / 'student_supervised_best.pth'
    history, best_acc = [], -1.0

    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        total_loss, correct, n = 0.0, 0, 0
        progress = tqdm(train_loader, desc=f'Supervised {epoch}/{STUDENT_EPOCHS}', leave=False)

        for images, labels in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            student_logits = student(images)
            loss = F.cross_entropy(student_logits, labels)

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            n += batch_size
            total_loss += loss.item() * batch_size
            correct += (student_logits.argmax(dim=1) == labels).sum().item()
            progress.set_postfix(loss=f'{total_loss/n:.3f}', acc=f'{correct/n:.2%}')

        best_acc = update_student_history(
            'supervised', epoch, student, history, best_acc, checkpoint,
            train_loss=total_loss/n, train_acc=correct/n,
        )
        scheduler.step()

    return restore_best_student('supervised', student, checkpoint, history, best_acc)


experiments['supervised'] = train_supervised_student()

## 7. Method 1: Soft-target knowledge distillation (Hinton KD)

A classifier produces **logits**: raw scores before applying softmax. We turn logits into class probabilities using **temperature** \(T\):

\[
p_t^{(T)} = \operatorname{softmax}(z_t / T), \qquad
p_s^{(T)} = \operatorname{softmax}(z_s / T).
\]

Here \(z_t\) and \(z_s\) are the teacher and student logits. With `T=1`, we obtain the usual probabilities. With a **larger temperature**, the probabilities are softer: the teacher's less likely classes become easier to see.

The classical KD loss combines two terms:

\[
\boxed{\mathcal L_{\mathrm{KD}}=
\alpha\,\underbrace{\mathrm{CE}(z_s,y)}_{\text{true labels}}
+(1-\alpha)T^2\underbrace{D_{\mathrm{KL}}\bigl(p_t^{(T)}\Vert p_s^{(T)}\bigr)}_{\text{teacher probabilities}}}
\]

- **Cross-entropy (CE):** encourages the student to predict the correct label.
- **KL divergence:** encourages the student to match the teacher's *whole class distribution*.
- **`alpha`:** determines how strongly we prioritize the original labels versus the teacher. In this notebook, `alpha=0.5` weights the two terms equally *before accounting for their numerical scales*.
- **`T²`:** the usual correction to the gradient scale when the temperature changes.

In PyTorch, `F.kl_div` expects **student log-probabilities as its first argument** and teacher probabilities as its second argument. `reduction='batchmean'` averages the sum over classes across the batch.


Reference: [Hinton, Vinyals & Dean, *Distilling the Knowledge in a Neural Network* (2015)](https://arxiv.org/abs/1503.02531).

In [ ]:
def hinton_kd_loss(student_logits, teacher_logits, targets, temperature=4.0, alpha=0.5):
    assert temperature > 0
    ce = F.cross_entropy(student_logits, targets)
    soft_teacher = F.softmax(teacher_logits.detach() / temperature, dim=1)
    log_soft_student = F.log_softmax(student_logits / temperature, dim=1)
    kd = F.kl_div(log_soft_student, soft_teacher, reduction='batchmean') * (temperature ** 2)
    return alpha * ce + (1 - alpha) * kd, {'ce': ce, 'kd': kd}

# Examine one illustrative softened distribution before training.
example_teacher_logits = torch.tensor([[5.0, 2.5, 0.2, -1.0]])
for temp in [1.0, 2.0, 4.0]:
    print(f'T={temp:g}:', torch.softmax(example_teacher_logits / temp, dim=1).round(decimals=3).tolist()[0])


In [ ]:
# Temperature changes the distribution without changing the class ranking.
toy_logits = torch.tensor([5.0, 2.5, 0.2, -1.0])
toy_labels = ['cat', 'dog', 'deer', 'truck']
fig, ax = plt.subplots(figsize=(8, 4))
for temperature in (1.0, 2.0, 4.0, 8.0):
    probs = F.softmax(toy_logits / temperature, dim=0).numpy()
    ax.plot(toy_labels, probs, marker='o', label=f'T = {temperature:g}')
ax.set(title='Effect of temperature on class probabilities',
       xlabel='Class', ylabel='Probability', ylim=(0, 1))
ax.legend()
ax.grid(alpha=0.2)
plt.show()

### Train the student with soft-target KD

This training loop resembles supervised learning, with two important additions:

1. The frozen teacher predicts probabilities for the same images, inside `torch.no_grad()`.
2. The student loss contains both ground-truth cross-entropy and a **temperature-scaled KL divergence** from the teacher distribution.

Inspect the lines that compute `teacher_probs`, `student_log_probs` and `kd_loss`. These are the essential parts of classical Hinton KD.

In [ ]:
def train_hinton_student():
    student = fresh_student()
    optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STUDENT_EPOCHS)
    checkpoint = CHECKPOINT_DIR / 'student_hinton_best.pth'
    history, best_acc = [], -1.0
    teacher.eval()

    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        total_loss, total_ce, total_kd, correct, n = 0.0, 0.0, 0.0, 0, 0
        progress = tqdm(train_loader, desc=f'Hinton KD {epoch}/{STUDENT_EPOCHS}', leave=False)

        for images, labels in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                teacher_logits = teacher(images)
            student_logits = student(images)

            ce_loss = F.cross_entropy(student_logits, labels)
            teacher_probs = F.softmax(teacher_logits / TEMPERATURE, dim=1)
            student_log_probs = F.log_softmax(student_logits / TEMPERATURE, dim=1)
            kd_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * TEMPERATURE**2
            loss = ALPHA * ce_loss + (1 - ALPHA) * kd_loss

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            n += batch_size
            total_loss += loss.item() * batch_size
            total_ce += ce_loss.item() * batch_size
            total_kd += kd_loss.item() * batch_size
            correct += (student_logits.argmax(dim=1) == labels).sum().item()
            progress.set_postfix(ce=f'{total_ce/n:.3f}', kd=f'{total_kd/n:.3f}', acc=f'{correct/n:.2%}')

        best_acc = update_student_history(
            'hinton', epoch, student, history, best_acc, checkpoint,
            train_loss=total_loss/n, train_acc=correct/n,
            ce=total_ce/n, kd=total_kd/n,
        )
        scheduler.step()

    return restore_best_student('hinton', student, checkpoint, history, best_acc)


experiments['hinton'] = train_hinton_student()

## 8. Method 2: Feature distillation

Instead of copying only the final prediction, we can ask the student to learn **internal image representations** similar to those of the teacher.

A convolutional feature map has shape `[B, C, H, W]`. Its channels can represent different image patterns, and its spatial dimensions preserve information about **where** those patterns are activated.

For our networks, the selected maps do not have the same dimensions:

```text
Student features                    Teacher features
[B, 128, 8, 8]                      [B, 512, 4, 4]
       │                                   ▲
       ▼                                   │
Average pooling (8×8 → 4×4)               │
       │                                   │
       ▼                                   │
1×1 convolution (128 → 512 channels) ─────┘
```

The `1×1` convolution is a small **trainable adapter**. It changes the number of channels without mixing neighbouring spatial positions. The average-pooling step matches the resolution.

We normalize the projected student map and the teacher map and then calculate their mean-squared error:

\[
\mathcal L_{\mathrm{feat}} = \frac{1}{N}\left\|\operatorname{Norm}(A(F_s))-
\operatorname{Norm}(F_t)\right\|_2^2,
\]

where \(A\) is the adapter and \(N\) is the number of values in the aligned feature tensor. In the implementation, `Norm` standardizes each image's feature map across channels and spatial positions.

The **training objective** is:

\[
\boxed{\mathcal L = \mathrm{CE}(z_s,y) + \lambda_f\mathcal L_{\mathrm{feat}}.}
\]

The student and adapter both receive gradients. The teacher remains frozen. **At inference time, we discard the adapter** and use the original small CNN, so this extra training component does not increase the deployed model's parameter count.

This is a simple **feature-regression** example. It is related to FitNets, but the original FitNets method includes a separate intermediate *hint training* stage.

Related work: [Romero et al., *FitNets: Hints for Thin Deep Nets* (2015)](https://arxiv.org/abs/1412.6550).

In [ ]:
class FeatureAdapter(nn.Module):
    '''Align the student's channel and spatial dimensions with the teacher's.'''
    def __init__(self, student_channels=128, teacher_channels=512):
        super().__init__()
        self.projection = nn.Conv2d(student_channels, teacher_channels, kernel_size=1)

    def forward(self, student_map, teacher_spatial_shape):
        student_map = F.adaptive_avg_pool2d(student_map, teacher_spatial_shape)
        return self.projection(student_map)


def standardize_feature_map(features):
    # Per-image normalization over C, H, W to compare activation patterns.
    mean = features.mean(dim=(1, 2, 3), keepdim=True)
    var = features.var(dim=(1, 2, 3), unbiased=False, keepdim=True)
    return (features - mean) / torch.sqrt(var + 1e-5)


def feature_regression_loss(aligned_student_features, teacher_features):
    target = teacher_features.detach()
    return F.mse_loss(standardize_feature_map(aligned_student_features),
                      standardize_feature_map(target))

# Sanity check the alignment dimensions and that the adapter receives gradients.
test_adapter = FeatureAdapter().to(device)
source_feat = torch.randn(2, 128, 8, 8, device=device, requires_grad=True)
target_feat = torch.randn(2, 512, 4, 4, device=device)
aligned = test_adapter(source_feat, target_feat.shape[-2:])
assert aligned.shape == target_feat.shape
feature_regression_loss(aligned, target_feat).backward()
assert test_adapter.projection.weight.grad is not None
print('Student:', tuple(source_feat.shape), '=> aligned:', tuple(aligned.shape))
del test_adapter, source_feat, target_feat, aligned


### Train the student with feature distillation

The teacher and student produce feature maps with different shapes. We use a **trainable adapter** to bring the student map to the same shape as the teacher map.

Notice three details in the code:

- The student forward pass returns both logits and features.
- The optimizer updates **the student and the adapter**. The teacher is frozen.
- The feature loss is added to cross-entropy. The adapter is used only during training, not during student inference.


In [ ]:
def train_feature_student():
    student = fresh_student()
    adapter = FeatureAdapter().to(device)
    optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(adapter.parameters()), lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STUDENT_EPOCHS)
    checkpoint = CHECKPOINT_DIR / 'student_feature_best.pth'
    history, best_acc = [], -1.0
    teacher.eval()

    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        adapter.train()
        total_loss, total_ce, total_feat, correct, n = 0.0, 0.0, 0.0, 0, 0
        progress = tqdm(train_loader, desc=f'Feature KD {epoch}/{STUDENT_EPOCHS}', leave=False)

        for images, labels in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                _, teacher_features = teacher_forward_with_features(teacher, images)
            student_logits, student_features = student.forward_with_features(images)
            aligned_features = adapter(student_features, teacher_features.shape[-2:])

            ce_loss = F.cross_entropy(student_logits, labels)
            feature_loss = feature_regression_loss(aligned_features, teacher_features)
            loss = ce_loss + FEATURE_WEIGHT * feature_loss

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            n += batch_size
            total_loss += loss.item() * batch_size
            total_ce += ce_loss.item() * batch_size
            total_feat += feature_loss.item() * batch_size
            correct += (student_logits.argmax(dim=1) == labels).sum().item()
            progress.set_postfix(ce=f'{total_ce/n:.3f}', feature=f'{total_feat/n:.3f}', acc=f'{correct/n:.2%}')

        best_acc = update_student_history(
            'feature', epoch, student, history, best_acc, checkpoint,
            train_loss=total_loss/n, train_acc=correct/n, adapter=adapter,
            ce=total_ce/n, feature=total_feat/n,
        )
        scheduler.step()

    return restore_best_student('feature', student, checkpoint, history, best_acc, adapter)


experiments['feature'] = train_feature_student()

## 9. Method 3: Logit matching with mean-squared error

Hinton KD compares **probability distributions** using KL divergence. Another option is to directly compare the **raw logits**, which are the scores produced before softmax.

\[
\mathcal L_{\mathrm{logit}} = \frac{1}{BC}\sum_{i=1}^{B}\sum_{c=1}^{C}\big(z_s[i,c]-z_t[i,c]\big)^2.
\]

Here, \(B\) is the batch size and \(C\) is the number of classes. We keep the standard cross-entropy term:

\[
\mathcal L = \mathrm{CE}(z_s,y) + \lambda_z\,\mathcal L_{\mathrm{logit}}.
\]

**What is different from Hinton KD?**

- Logit MSE matches the **numerical scores**, not just the probabilities.
- It does not need a temperature parameter.
- It is sensitive to the **scale and offset** of the logits. Adding the same constant to all logits does not change their softmax probabilities, but it *does* change the MSE.
- The MSE can be large when the teacher is very confident. The coefficient `LOGIT_WEIGHT` must be selected carefully.

Thus, logit matching is simple, but it is not automatically better than KL-based KD. The next visualization demonstrates the difference.


In [ ]:
def logit_matching_loss(student_logits, teacher_logits, targets, logit_weight=0.05):
    ce = F.cross_entropy(student_logits, targets)
    logit_mse = F.mse_loss(student_logits, teacher_logits.detach())
    return ce + logit_weight * logit_mse, {'ce': ce, 'logit_mse': logit_mse}


# Adding the same constant to every logit leaves the probability distribution unchanged.
logits_a = torch.tensor([[2.0, 1.0, 0.0]])
logits_b = logits_a + 4.0
print('Softmax equal:', torch.allclose(F.softmax(logits_a, dim=1),
                                    F.softmax(logits_b, dim=1)))
print('MSE between logits:', F.mse_loss(logits_a, logits_b).item())


In [ ]:
# Both sets of logits yield the same probabilities, yet their values differ.
class_names = ['cat', 'dog', 'deer']
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(np.arange(3)-0.18, logits_a[0].numpy(), width=0.35, label='Logits A')
axes[0].bar(np.arange(3)+0.18, logits_b[0].numpy(), width=0.35, label='Logits B')
axes[0].set_xticks(range(3), class_names)
axes[0].set_ylabel('Raw score')
axes[0].set_title('Different logits')
axes[0].legend()
axes[1].plot(class_names, F.softmax(logits_a, dim=1)[0].numpy(), 'o-', label='A')
axes[1].plot(class_names, F.softmax(logits_b, dim=1)[0].numpy(), 'x--', label='B')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Probability')
axes[1].set_title('Identical probability distributions')
axes[1].legend()
plt.tight_layout()
plt.show()


### Train a student using logit regression

Read the training loop below. The teacher produces **raw class scores**, not pseudo-labels. The student receives a gradient from both `F.cross_entropy()` and `F.mse_loss()`.

Compare the two objectives in the previous sections:

- Hinton KD: `F.kl_div()` on soft probabilities.
- Logit regression: `F.mse_loss()` on unnormalized logits.

The student and teacher must output the same number of classes.

In [ ]:
def train_logit_student():
    student = fresh_student()
    optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STUDENT_EPOCHS)
    checkpoint = CHECKPOINT_DIR / 'student_logit_best.pth'
    history, best_acc = [], -1.0
    teacher.eval()

    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        total_loss, total_ce, total_logits, correct, n = 0.0, 0.0, 0.0, 0, 0
        progress = tqdm(train_loader, desc=f'Logit MSE {epoch}/{STUDENT_EPOCHS}', leave=False)

        for images, labels in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                teacher_logits = teacher(images)
            student_logits = student(images)

            ce_loss = F.cross_entropy(student_logits, labels)
            logit_loss = F.mse_loss(student_logits, teacher_logits)
            loss = ce_loss + LOGIT_WEIGHT * logit_loss

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            n += batch_size
            total_loss += loss.item() * batch_size
            total_ce += ce_loss.item() * batch_size
            total_logits += logit_loss.item() * batch_size
            correct += (student_logits.argmax(dim=1) == labels).sum().item()
            progress.set_postfix(ce=f'{total_ce/n:.3f}', logit_mse=f'{total_logits/n:.3f}',
                                 acc=f'{correct/n:.2%}')

        best_acc = update_student_history(
            'logit', epoch, student, history, best_acc, checkpoint,
            train_loss=total_loss/n, train_acc=correct/n,
            ce=total_ce/n, logit_mse=total_logits/n,
        )
        scheduler.step()

    return restore_best_student('logit', student, checkpoint, history, best_acc)


experiments['logit'] = train_logit_student()


## 10. Combine probability and feature distillation

It is also possible to ask the student to match **both** the teacher's class probabilities and its internal feature map:

\[
\mathcal L_{\mathrm{combined}}
= \alpha\,\mathcal L_{\mathrm{CE}}
+ (1-\alpha) T^2 D_{\mathrm{KL}}(p_t^{(T)}\Vert p_s^{(T)})
+ \lambda_f\,\mathcal L_{\mathrm{feat}}.
\]

This is not guaranteed to be better. The three terms have different numerical scales and may compete during optimization.

The next function has a **separate training procedure**, including the full loss expression. Set `RUN_COMBINED = True` in the settings to run it.

In [ ]:
def train_combined_student():
    student = fresh_student()
    adapter = FeatureAdapter().to(device)
    optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(adapter.parameters()), lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STUDENT_EPOCHS)
    checkpoint = CHECKPOINT_DIR / 'student_combined_best.pth'
    history, best_acc = [], -1.0
    teacher.eval()

    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        adapter.train()
        total_loss, total_ce, total_kd, total_feat, correct, n = 0.0, 0.0, 0.0, 0.0, 0, 0
        progress = tqdm(train_loader, desc=f'Combined KD {epoch}/{STUDENT_EPOCHS}', leave=False)

        for images, labels in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                teacher_logits, teacher_features = teacher_forward_with_features(teacher, images)
            student_logits, student_features = student.forward_with_features(images)
            aligned_features = adapter(student_features, teacher_features.shape[-2:])

            ce_loss = F.cross_entropy(student_logits, labels)
            teacher_probs = F.softmax(teacher_logits / TEMPERATURE, dim=1)
            student_log_probs = F.log_softmax(student_logits / TEMPERATURE, dim=1)
            kd_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * TEMPERATURE**2
            feature_loss = feature_regression_loss(aligned_features, teacher_features)
            loss = ALPHA * ce_loss + (1 - ALPHA) * kd_loss + FEATURE_WEIGHT * feature_loss

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            n += batch_size
            total_loss += loss.item() * batch_size
            total_ce += ce_loss.item() * batch_size
            total_kd += kd_loss.item() * batch_size
            total_feat += feature_loss.item() * batch_size
            correct += (student_logits.argmax(dim=1) == labels).sum().item()
            progress.set_postfix(loss=f'{total_loss/n:.3f}', acc=f'{correct/n:.2%}')

        best_acc = update_student_history(
            'combined', epoch, student, history, best_acc, checkpoint,
            train_loss=total_loss/n, train_acc=correct/n, adapter=adapter,
            ce=total_ce/n, kd=total_kd/n, feature=total_feat/n,
        )
        scheduler.step()

    return restore_best_student('combined', student, checkpoint, history, best_acc, adapter)


if RUN_COMBINED:
    experiments['combined'] = train_combined_student()

## 11. Evaluate and compare

**Accuracy** is the fraction of correctly classified images. We select checkpoints using the validation set, then calculate accuracy on the separate **test set**.

The table includes **inference parameter counts**. Feature distillation introduces an adapter *during training*, but the adapter is not part of the final classifier. All student methods therefore have the same inference size.

The two plots show:

1. **Validation accuracy:** how well the classifiers generalize across epochs.
2. **Validation cross-entropy:** one common metric that *can* be compared across methods (unlike their different training losses).

Distillation does not always improve accuracy. Consider teacher quality, optimization, and loss weights before drawing conclusions from one run.

In [ ]:
results = []
teacher_test = run_supervised_epoch(teacher, test_loader)
results.append({'method': 'teacher (reference)', 'best val acc (%)': 100*teacher_val['acc'],
                'test acc (%)': 100*teacher_test['acc'], 'inference params': count_parameters(teacher)})
for name, item in experiments.items():
    test_metrics = run_supervised_epoch(item['student'], test_loader)
    results.append({'method': name, 'best val acc (%)': 100*item['best_val_acc'],
                    'test acc (%)': 100*test_metrics['acc'],
                    'inference params': count_parameters(item['student'])})

results_df = pd.DataFrame(results).set_index('method')
display(results_df.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, item in experiments.items():
    hist = item['history']
    axes[0].plot(hist['epoch'], hist['val_acc'] * 100, marker='o', label=name)
    axes[1].plot(hist['epoch'], hist['val_loss'], marker='o', label=name)
axes[0].axhline(100*teacher_val['acc'], linestyle='--', label='teacher', color='black')
axes[0].set(title='Validation accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
axes[1].set(title='Validation cross-entropy', xlabel='Epoch', ylabel='Cross-entropy')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


## 12. Inspect the class probabilities

We compare the teacher with the students on **one held-out image**, using the same temperature for every model.

In [ ]:
def unnormalize_image(img):
    mean = torch.tensor(MEAN, device=img.device).view(3, 1, 1)
    std = torch.tensor(STD, device=img.device).view(3, 1, 1)
    return (img * std + mean).permute(1, 2, 0).clamp(0, 1).cpu().numpy()


def show_soft_predictions(teacher, experiments, loader, image_index=0, temperature=4.0):
    xb, yb = next(iter(loader))
    img = xb[image_index:image_index+1].to(device)
    with torch.no_grad():
        teacher_probs = F.softmax(teacher(img) / temperature, dim=1)[0].cpu().numpy()
        student_probs = {
            method: F.softmax(item['student'](img) / temperature, dim=1)[0].cpu().numpy()
            for method, item in experiments.items()
        }
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    axs[0].imshow(unnormalize_image(xb[image_index]))
    axs[0].set(title=f'Input / true class: {classes[yb[image_index].item()]}')
    axs[0].axis('off')
    x = np.arange(len(classes))
    axs[1].plot(x, teacher_probs, marker='o', linewidth=2.5, label='teacher')
    for name, probabilities in student_probs.items():
        axs[1].plot(x, probabilities, marker='.', label=name)
    axs[1].set_xticks(x, classes, rotation=45, ha='right')
    axs[1].set(title=f'Class probabilities at T={temperature}', ylabel='Probability', ylim=(0, 1))
    axs[1].grid(alpha=0.2)
    axs[1].legend()
    plt.tight_layout()
    plt.show()

show_soft_predictions(teacher, experiments, test_loader, image_index=0, temperature=TEMPERATURE)


## 13. Inspect the feature maps

A feature map contains many channels, so we cannot visualize every activation in a single image. Instead, the next cell shows **spatial activation energy**: the average squared activation at each location, after standardizing the maps.

We compare the teacher map with the student map **after alignment by the adapter**. Similar-looking heatmaps suggest that both networks emphasize similar image regions, but this is only a visual illustration. The feature loss compares the **full feature tensors**, not just the heatmap.


In [ ]:
def show_feature_maps(teacher, experiment, loader, image_index=0):
    if experiment['adapter'] is None:
        print('Choose an experiment trained with feature alignment (feature or combined).')
        return
    xb, yb = next(iter(loader))
    with torch.no_grad():
        img = xb[image_index:image_index+1].to(device)
        _, tmap = teacher_forward_with_features(teacher, img)
        _, smap = experiment['student'].forward_with_features(img)
        aligned_smap = experiment['adapter'](smap, tmap.shape[-2:])
        tmap = standardize_feature_map(tmap)
        aligned_smap = standardize_feature_map(aligned_smap)
        heat_t = tmap.square().mean(dim=1)[0].cpu().numpy()
        heat_s = aligned_smap.square().mean(dim=1)[0].cpu().numpy()
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
    axs[0].imshow(unnormalize_image(xb[image_index]))
    axs[0].set_title(classes[yb[image_index].item()])
    for ax, array, title in [(axs[1], heat_t, 'Teacher feature energy'),
                             (axs[2], heat_s, 'Aligned student feature energy')]:
        ax.imshow(array, cmap='magma')
        ax.set_title(title)
    for ax in axs:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

if 'feature' in experiments:
    show_feature_maps(teacher, experiments['feature'], test_loader)
elif 'combined' in experiments:
    show_feature_maps(teacher, experiments['combined'], test_loader)


## 14. Questions for the CIFAR-10 experiments

1. **Teacher quality:** what happens if the teacher is not better than the baseline student?
2. **Temperature:** try `TEMPERATURE = 1` and `8`. How does the teacher's distribution change?
3. **Logit scale:** why can logit MSE be nonzero when two models have exactly the same softmax probabilities? Try `LOGIT_WEIGHT = 0.01` or `0.2`.
4. **Feature weights:** what might happen if `FEATURE_WEIGHT` is much larger than cross-entropy?
5. **Combined KD:** enable `RUN_COMBINED`. Does matching both logits and features help in your experiment?
6. **Experimental design:** why should all students use the same training set, number of epochs, and checkpoint rule?

Change one setting at a time. Since training is stochastic, repeat runs with different seeds before drawing strong conclusions.

---

# Part II. Knowledge distillation for regression

Until now, we predicted **classes**. What if the target is a continuous number, such as temperature, distance, or energy consumption?

In a regression task, a model predicts \(\hat y \in \mathbb{R}\), not a probability distribution over classes. We cannot use classification cross-entropy or an `argmax` pseudo-label. Instead, a natural choice is **mean-squared error (MSE)**.

We will use a small synthetic dataset so that the true curve can be visualized. The observed labels contain noise, and the student sees relatively few labeled examples. The teacher can also generate target values at **additional unlabeled inputs**.


## 15. Regression dataset and models

We observe a scalar input \(x\) and a noisy continuous label \(y\). Our larger teacher uses two hidden layers of width 64. The student uses two hidden layers of width 12.

Besides the 60 labeled training points, we sample more inputs without labels. Those extra inputs will be used only for distillation. This makes the role of the teacher easy to see: it supplies additional **continuous targets**, not ground-truth observations.

In [ ]:
# This section is independent from CIFAR-10 and runs quickly on CPU.
reg_rng = torch.Generator().manual_seed(2026)

def true_regression_function(x):
    return torch.sin(2.3 * x) + 0.25 * x

x_reg_train = torch.linspace(-3.0, 3.0, 60).unsqueeze(1)
y_reg_train = true_regression_function(x_reg_train) + 0.15 * torch.randn(
    x_reg_train.shape, generator=reg_rng
)
x_reg_unlabeled = 6.0 * torch.rand(500, 1, generator=reg_rng) - 3.0
x_reg_test = torch.linspace(-3.0, 3.0, 300).unsqueeze(1)
y_reg_test = true_regression_function(x_reg_test)

# Keep the regression example on CPU for a predictable, lightweight demo.
reg_device = torch.device('cpu')
x_reg_train, y_reg_train = x_reg_train.to(reg_device), y_reg_train.to(reg_device)
x_reg_unlabeled = x_reg_unlabeled.to(reg_device)
x_reg_test, y_reg_test = x_reg_test.to(reg_device), y_reg_test.to(reg_device)

class RegressionTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


class RegressionStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 12), nn.Tanh(),
            nn.Linear(12, 12), nn.Tanh(),
            nn.Linear(12, 1),
        )

    def forward(self, x):
        return self.net(x)


fig, ax = plt.subplots(figsize=(9, 3.5))
ax.scatter(x_reg_train.squeeze().numpy(), y_reg_train.squeeze().numpy(),
           s=22, alpha=0.6, label='Noisy labeled examples')
ax.plot(x_reg_test.squeeze().numpy(), y_reg_test.squeeze().numpy(),
        color='black', label='True curve (evaluation only)')
ax.set(xlabel='x', ylabel='y', title='Regression data')
ax.legend()
plt.show()


## 16. Train the regression teacher

The teacher receives the noisy labeled pairs \((x_i, y_i)\) and minimizes:

\[
\mathcal L_{\mathrm{teacher}} = \frac{1}{N}\sum_i \big(f_t(x_i) - y_i\big)^2.
\]

It will be frozen after training. We use a small amount of weight decay to discourage unnecessarily large parameters. This does not guarantee a better teacher, so always inspect its test error.

In [ ]:
torch.manual_seed(2026)
reg_teacher = RegressionTeacher().to(reg_device)
reg_teacher_optimizer = torch.optim.AdamW(reg_teacher.parameters(), lr=0.01, weight_decay=1e-3)

for epoch in range(450):
    reg_teacher.train()
    reg_teacher_optimizer.zero_grad(set_to_none=True)
    predicted_y = reg_teacher(x_reg_train)
    teacher_loss = F.mse_loss(predicted_y, y_reg_train)
    teacher_loss.backward()
    reg_teacher_optimizer.step()
    if (epoch + 1) % 150 == 0:
        print(f'Teacher epoch {epoch+1:3d} | train MSE: {teacher_loss.item():.4f}')

reg_teacher.eval()
reg_teacher.requires_grad_(False)
with torch.no_grad():
    reg_teacher_test_mse = F.mse_loss(reg_teacher(x_reg_test), y_reg_test).item()
print(f'Teacher clean test MSE: {reg_teacher_test_mse:.4f}')


## 17. Regression baseline: train a student without distillation

First, train the small model **only on observed labels**. This is our supervised baseline:

\[
\mathcal L_{\mathrm{supervised}} = \operatorname{MSE}(f_s(x_{\mathrm{labeled}}), y_{\mathrm{labeled}}).
\]

Store the student's initial weights so that the distilled student can start from the same point.

In [ ]:
torch.manual_seed(2027)
initial_reg_student = RegressionStudent()
initial_reg_student_state = {k: v.detach().clone() for k, v in initial_reg_student.state_dict().items()}

reg_student_supervised = RegressionStudent().to(reg_device)
reg_student_supervised.load_state_dict(initial_reg_student_state)
reg_student_optimizer = torch.optim.AdamW(reg_student_supervised.parameters(), lr=0.01)
reg_supervised_history = []

for epoch in range(300):
    reg_student_supervised.train()
    reg_student_optimizer.zero_grad(set_to_none=True)
    student_prediction = reg_student_supervised(x_reg_train)
    supervised_loss = F.mse_loss(student_prediction, y_reg_train)
    supervised_loss.backward()
    reg_student_optimizer.step()
    reg_supervised_history.append(supervised_loss.item())
    if (epoch + 1) % 100 == 0:
        print(f'Supervised student epoch {epoch+1:3d} | train MSE: {supervised_loss.item():.4f}')

reg_student_supervised.eval()


## 18. Regression student: distill continuous teacher predictions

Now train the same student architecture again, from **identical initial weights**. The supervised MSE remains, but we add a second term:

\[
\mathcal L_{\mathrm{KD-reg}} = \operatorname{MSE}\big(f_s(x_{\mathrm{unlabeled}}), f_t(x_{\mathrm{unlabeled}})\big).
\]

The full objective is:

\[
\boxed{\mathcal L = \operatorname{MSE}(f_s(x_L), y_L)
+ \lambda\operatorname{MSE}(f_s(x_U), f_t(x_U)).}
\]

`x_L` has observed labels. `x_U` contains additional inputs **without observed labels**. The frozen teacher provides the numerical targets \(f_t(x_U)\). We use `torch.no_grad()` for the teacher to ensure that only the student is updated.

**Why can this help?** The teacher may approximate the underlying relationship well and provide a smoother training signal at more input locations. **Why can it fail?** When the teacher is inaccurate, distillation may transfer its errors. The loss weight \(\lambda\) controls the tradeoff.

In [ ]:
REGRESSION_KD_WEIGHT = 0.5

reg_student_distilled = RegressionStudent().to(reg_device)
reg_student_distilled.load_state_dict(initial_reg_student_state)
reg_kd_optimizer = torch.optim.AdamW(reg_student_distilled.parameters(), lr=0.01)
reg_kd_history = []

reg_teacher.eval()
for epoch in range(300):
    reg_student_distilled.train()
    reg_kd_optimizer.zero_grad(set_to_none=True)

    # MSE using true observed labels.
    student_labeled = reg_student_distilled(x_reg_train)
    supervised_loss = F.mse_loss(student_labeled, y_reg_train)

    # MSE using teacher predictions on additional unlabeled inputs.
    with torch.no_grad():
        teacher_unlabeled = reg_teacher(x_reg_unlabeled)
    student_unlabeled = reg_student_distilled(x_reg_unlabeled)
    distillation_loss = F.mse_loss(student_unlabeled, teacher_unlabeled)

    loss = supervised_loss + REGRESSION_KD_WEIGHT * distillation_loss
    loss.backward()
    reg_kd_optimizer.step()
    reg_kd_history.append((supervised_loss.item(), distillation_loss.item()))
    if (epoch + 1) % 100 == 0:
        print(f'Regression KD epoch {epoch+1:3d} | '
              f'labeled MSE: {supervised_loss.item():.4f} | '
              f'teacher MSE: {distillation_loss.item():.4f}')

reg_student_distilled.eval()


## 19. Compare the regression models

All three models predict values for the **same held-out inputs**. We compare them with clean targets using **MSE** and **RMSE**. Lower values indicate better predictions.

The true curve is available here because the data are synthetic. On real regression datasets, we would instead use a held-out set of labeled examples.

In [ ]:
with torch.no_grad():
    reg_predictions = {
        'Teacher': reg_teacher(x_reg_test).squeeze().numpy(),
        'Supervised student': reg_student_supervised(x_reg_test).squeeze().numpy(),
        'Distilled student': reg_student_distilled(x_reg_test).squeeze().numpy(),
    }

ground_truth = y_reg_test.squeeze().numpy()
reg_results = []
for name, predictions in reg_predictions.items():
    mse = float(np.mean((predictions - ground_truth) ** 2))
    reg_results.append({'model': name, 'test MSE': mse, 'test RMSE': np.sqrt(mse)})
display(pd.DataFrame(reg_results).set_index('model').round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(x_reg_train.squeeze().numpy(), y_reg_train.squeeze().numpy(),
                s=15, alpha=0.35, label='Noisy labels')
axes[0].plot(x_reg_test.squeeze().numpy(), ground_truth, color='black',
             linewidth=2, label='True curve')
for name, predictions in reg_predictions.items():
    axes[0].plot(x_reg_test.squeeze().numpy(), predictions, label=name)
axes[0].set(title='Continuous predictions', xlabel='x', ylabel='y')
axes[0].legend(fontsize=8)

axes[1].plot(reg_supervised_history, label='Supervised student: labeled MSE')
axes[1].plot([v[0] for v in reg_kd_history], label='Distilled student: labeled MSE')
axes[1].plot([v[1] for v in reg_kd_history], label='Distilled student: teacher MSE')
axes[1].set(title='Student training losses', xlabel='Epoch', ylabel='MSE')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
